# 04 — Retrieval-Augmented Context — Practical Examples

**Covers**: OpenAI embeddings, cosine similarity, in-memory vector store, RAGContextManager, comparison with sliding window

In [ ]:
import os, json
import numpy as np
import tiktoken
from openai import OpenAI
from dataclasses import dataclass, field
from typing import Optional
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv()
client = OpenAI()
MODEL        = 'gpt-4o-mini'
EMBED_MODEL  = 'text-embedding-3-small'
enc = tiktoken.encoding_for_model(MODEL)
print('✅ Setup complete')

## 1. Embedding Text — The Foundation

In [ ]:
def embed_text(text: str) -> list[float]:
    r = client.embeddings.create(model=EMBED_MODEL, input=text.replace('\n', ' '))
    return r.data[0].embedding

def cosine_similarity(a: list[float], b: list[float]) -> float:
    va, vb = np.array(a), np.array(b)
    return float(np.dot(va, vb) / (np.linalg.norm(va) * np.linalg.norm(vb)))

# Show how similar/different texts are in embedding space
sentences = [
    "Python list comprehensions are a concise way to create lists.",
    "You can create lists efficiently using list comprehensions in Python.",  # Similar
    "FastAPI is a modern web framework for building APIs.",                   # Different
    "LLMs have a limited context window measured in tokens.",                  # Very different
]

embeddings = [embed_text(s) for s in sentences]
ref = embeddings[0]

print(f"Reference: {sentences[0]!r}\n")
for i, (sent, emb) in enumerate(zip(sentences[1:], embeddings[1:]), 1):
    sim = cosine_similarity(ref, emb)
    bar = '█' * int(sim * 20)
    print(f"Sim={sim:.3f} {bar:<20} {sent[:60]}")

## 2. In-Memory Vector Store

In [ ]:
@dataclass
class MessageRecord:
    role: str
    content: str
    embedding: list[float]
    turn: int

class SimpleVectorStore:
    def __init__(self):
        self.records: list[MessageRecord] = []
        self._turn = 0
    
    def add(self, role: str, content: str):
        emb  = embed_text(content)
        self.records.append(MessageRecord(role, content, emb, self._turn))
        self._turn += 1
    
    def search(self, query: str, top_k: int = 3, min_sim: float = 0.65) -> list[MessageRecord]:
        if not self.records: return []
        qv = np.array(embed_text(query))
        scored = [(cosine_similarity(qv.tolist(), r.embedding), r) for r in self.records]
        scored.sort(key=lambda x: x[0], reverse=True)
        return [r for sim, r in scored[:top_k] if sim >= min_sim]

# Build a store with diverse conversation history
store = SimpleVectorStore()
history_entries = [
    ('user',      'What is a Python decorator?'),
    ('assistant', 'Decorators modify functions. Use @decorator syntax above the function definition.'),
    ('user',      'How do I handle errors in Python?'),
    ('assistant', 'Use try/except blocks. Catch specific exceptions for better error handling.'),
    ('user',      'Explain async/await in Python.'),
    ('assistant', 'async def creates a coroutine. await pauses execution until the awaited task completes.'),
    ('user',      'What is the GIL?'),
    ('assistant', 'The Global Interpreter Lock prevents true multi-threading in CPython. Use multiprocessing for CPU tasks.'),
]
for role, content in history_entries:
    store.add(role, content)

print(f"Stored {len(store.records)} messages\n")

# Test retrieval
query = 'How do exception handling work?'
results = store.search(query, top_k=2)
print(f"Query: {query!r}")
print(f"Top {len(results)} matches:")
for r in results:
    print(f"  [{r.role}] turn={r.turn}: {r.content[:70]}")

## 3. RAGContextManager in Action

In [ ]:
class RAGContextManager:
    def __init__(self, system: str, recent_turns: int = 3, retrieved_k: int = 2, min_sim: float = 0.68):
        self.system = system
        self.recent_turns = recent_turns
        self.retrieved_k = retrieved_k
        self.min_sim = min_sim
        self.vector_store = SimpleVectorStore()
        self._all: list[dict] = []
    
    def add(self, role: str, content: str):
        self._all.append({'role': role, 'content': content})
        self.vector_store.add(role, content)
    
    def get_messages(self, current_query: str) -> list[dict]:
        msgs = [{'role': 'system', 'content': self.system}]
        # Retrieve relevant historical context
        if len(self._all) > self.recent_turns * 2:
            retrieved = self.vector_store.search(current_query, self.retrieved_k, self.min_sim)
            if retrieved:
                ctx_text = '\n'.join(f"{r.role.upper()}: {r.content}" for r in retrieved)
                msgs += [
                    {'role': 'user',      'content': f'[Relevant past context]\n{ctx_text}'},
                    {'role': 'assistant', 'content': 'I recall this relevant context.'}
                ]
        # Always include recent raw history
        msgs += self._all[-(self.recent_turns * 2):]
        return msgs

# Run a long Q&A session using RAG context
rage_ctx = RAGContextManager(
    system='You are a Python expert tutor.',
    recent_turns=2, retrieved_k=2
)

qa_pairs = [
    ('What is a list comprehension?',   None),
    ('How does error handling work?',   None),
    ('Explain generators.',             None),
    ('What are context managers?',      None),
    # Now ask something that needs early context
    ('Give me an example of list comprehension with a filter condition.', None),
]

for question, _ in qa_pairs:
    rage_ctx.add('user', question)
    msgs = rage_ctx.get_messages(question)
    r = client.chat.completions.create(model=MODEL, messages=msgs, max_tokens=100)
    answer = r.choices[0].message.content
    rage_ctx.add('assistant', answer)
    import tiktoken
    ec = tiktoken.encoding_for_model(MODEL)
    toks = sum(3 + len(ec.encode(str(m.get('content',''))))
               for m in msgs) + 3
    print(f"Q: {question[:55]:<57} | Context: {len(msgs)} msgs, {toks} tokens")

## 4. Compare: RAG vs Sliding Window Token Usage

In [ ]:
# Simulate 20 turns and compare token usage growth
ec = tiktoken.encoding_for_model(MODEL)

raw_history = [{'role': 'system', 'content': 'You are a Python tutor.'}]

print(f"{'Turn':<6} {'Raw (all msgs)':>16} {'Sliding (8 msg)':>16} {'RAG (2+2 msgs)':>16}")
print('-' * 60)

for turn in range(1, 12):
    user_msg = {'role': 'user',      'content': f'Question about Python topic {turn}: explain concept {turn}.'}
    asst_msg = {'role': 'assistant', 'content': f'Answer for topic {turn}: This concept {turn} works by doing X, Y, Z in Python.'}
    raw_history.extend([user_msg, asst_msg])
    
    raw_tokens  = sum(3 + len(ec.encode(str(m.get('content','')))) for m in raw_history) + 3
    
    # Sliding: keep system + last 8 messages
    slid = [raw_history[0]] + raw_history[-8:] if len(raw_history) > 9 else raw_history
    slid_tokens = sum(3 + len(ec.encode(str(m.get('content','')))) for m in slid) + 3
    
    # RAG: system + 2 retrieved (simulated as ~120 tokens) + 4 recent
    rag_msgs   = [raw_history[0]] + raw_history[-4:]
    rag_tokens = sum(3 + len(ec.encode(str(m.get('content','')))) for m in rag_msgs) + 3 + 120  # +120 for retrieved
    
    print(f"{turn:<6} {raw_tokens:>16,} {slid_tokens:>16,} {rag_tokens:>16,}")

## 📌 Summary

- **Embed** every message with `text-embedding-3-small` (~$0.02/1M tokens)
- **Cosine similarity** measures how semantically related two texts are
- **SimpleVectorStore**: add() → embed; search(query) → top-K similar
- **RAGContextManager**: system + retrieved relevant + recent raw = near-constant tokens
- **Best for**: long sessions, heterogeneous topics, knowledge-intensive agents
- **Production DB**: use ChromaDB, Qdrant, or Pinecone for persistence and scale